# Zebrafish embryogenesis

This notebook starts with raw counts, trains CytoBridge, and opens the resulting
growth, trajectory, and interaction analyses. Run the cells in order after
[installing CytoBridge](../../installation.md) and adding the input files below.
Training requires a GPU and is not run when this website is built.

To draw a figure from the paper's saved results without training, go directly
to [Paper figures](../paper_figures/index.md). Those results are separate from
the new model fitted here.

## Get the data

Use `data/zebrafish_raw.h5ad` for the counts H5AD. The [download
guide](../../data_checkpoints.md) lists the original study and the files still
needed for the paper's exact input. A study download may need conversion to
H5AD before it can be used here.

Run this notebook from your own working directory. All paths below are relative
to that directory. Use a new output folder for each training run.

## Set the paths

In [ ]:
from pathlib import Path
import json

from CytoBridge.workflow import WorkflowOptions, load_workflow_config, run_workflow

DATASET_CONFIG = 'zebrafish'
RAW_H5AD = Path("data/zebrafish_raw.h5ad")
OUTPUT_DIR = Path("outputs/zebrafish")
ALIGNED_H5AD = OUTPUT_DIR / "preprocess" / "zebrafish_aligned.h5ad"
MODEL_DIR = OUTPUT_DIR / "training"

config, _ = load_workflow_config(DATASET_CONFIG)

## Prepare the input

The configuration gives the names of the count layer, time column, cell-type
column, and spatial coordinates. This table shows the fields expected in the
raw H5AD.

In [ ]:
import pandas as pd

preprocess = config["preprocess"]
align = preprocess["align"]
coordinate_columns = align.get("spatial_obs_keys")
spatial_source = (
    f"obs[{coordinate_columns!r}]" if coordinate_columns
    else f"obsm[{align.get('input_spatial_key', 'spatial')!r}]"
)
pd.DataFrame({
    "Input": ["Counts", "Time", "Cell type", "Coordinates"],
    "AnnData field": [
        f"layers[{align.get('expression_layer', 'X')!r}]"
        if align.get("expression_layer", "X") != "X" else "X",
        f"obs[{preprocess['time_key']!r}]",
        f"obs[{preprocess['annotation_source']!r}]",
        spatial_source,
    ],
})

In [ ]:
if not RAW_H5AD.is_file():
    raise FileNotFoundError(f"Add the input H5AD or update RAW_H5AD: {RAW_H5AD}")

## Train and calculate the results

This call performs preprocessing, fits the LR edge predictor, trains the
dynamical model, and runs the configured downstream analyses. **Do not run a
separate preprocessing command first.**

The aligned data are written to `ALIGNED_H5AD`. Training reads that file and
writes the model to `MODEL_DIR`. Downstream analysis then reads both.

In [ ]:
result = run_workflow(
    config,
    options=WorkflowOptions(
        input_h5ad=RAW_H5AD,
        output_dir=OUTPUT_DIR,
        train=True,
        device="cuda",
    ),
)

The equivalent terminal command is below. Use either the Python call above
or this command, not both.

```bash
cytobridge workflow --config zebrafish --train \
  --input-h5ad data/zebrafish_raw.h5ad \
  --output-dir outputs/zebrafish --device cuda
```

## Open the results

The previous step creates the following files. The exact set of analyses is
selected by the dataset configuration.

| Result | Location under `OUTPUT_DIR` |
| --- | --- |
| Aligned expression and coordinates | `preprocess/` |
| Trained model and settings | `training/` |
| Observed and interpolated cell populations | `downstream/slice_data/` |
| Growth rates | `downstream/growth/` |
| Velocity components | `downstream/velocity/` |
| Cell-type interaction summaries | `downstream/communication/` |
| Plots | `downstream/figures/` |

Read the summary written by **this run**, then list its generated plots:

In [ ]:
downstream_dir = OUTPUT_DIR / "downstream"
summary = json.loads((downstream_dir / "summary.json").read_text())
summary

In [ ]:
figure_files = sorted((downstream_dir / "figures").rglob("*.png"))
pd.DataFrame({"Figure": [str(path.relative_to(OUTPUT_DIR)) for path in figure_files]})

To view one of these plots in the notebook:

In [ ]:
from IPython.display import Image, display

if figure_files:
    display(Image(filename=str(figure_files[0])))

## Paper figures

The plots above use your new model. The paper figure notebooks below explain
which saved numerical results they use and whether further calculations are
needed. They do not automatically use `OUTPUT_DIR`.

- [Supplementary Figures S31–S38](../paper_figures/zebrafish_si_s31_s38.ipynb)
- [Supplementary Figure S39](../paper_figures/zebrafish_attention.ipynb)
- [Supplementary Figure S40](../paper_figures/zebrafish_decomposition_stability.md)

For the calculation steps behind each panel, see the
[figure-by-figure guide](../../paper_reproduction.md).
To repeat analysis with an existing model, see
[Continue from a trained model](../../reuse_model.md).